# Gradient Boosting Classifier — Practical Notebook

## Hyperparameters from the supplied cheat sheet
The cheat sheet's **Gradient Boosting** grid specifies exactly:

- `n_estimators`: `[50, 100, 200]`
- `learning_rate`: `[0.01, 0.1, 0.2]`
- `max_depth`: `[1, 3, 5]`
- `subsample`: `[0.8, 1.0]`

That gives **54 parameter combinations**. The notebook fixes `random_state=42` for reproducibility. Other parameters such as `min_samples_split`, `min_samples_leaf`, and `max_features` are left at scikit-learn defaults because the cheat sheet does not give search values for them.

> This notebook is for **classification**. Select the target column when prompted. It is syntax-checked, but results will only appear after you run it with your dataset.

## 1. Imports and reproducibility
Run the notebook from the folder containing your CSV dataset. If there are multiple CSV files, the loader will ask you to choose one.

In [ ]:

import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, label_binarize
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, average_precision_score,
    log_loss, RocCurveDisplay, PrecisionRecallDisplay
)
from scipy.stats import ttest_rel, wilcoxon

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded. Random state:", RANDOM_STATE)


## 2. Load the dataset and choose the target column

In [ ]:

csv_files = sorted(glob.glob("*.csv"))

if not csv_files:
    file_path = input("Enter the path to your CSV file: ").strip().strip('"')
elif len(csv_files) == 1:
    file_path = csv_files[0]
    print("Using CSV:", file_path)
else:
    print("CSV files found:")
    for i, name in enumerate(csv_files):
        print(f"{i}: {name}")
    choice = input("Enter the number of the CSV to use: ").strip()
    file_path = csv_files[int(choice)]

df = pd.read_csv(file_path)
print("Dataset shape:", df.shape)
display(df.head())

print("\nColumns:")
for col in df.columns:
    print("-", col)

target_col = input("\nEnter the exact target column name: ").strip()
if target_col not in df.columns:
    raise ValueError(f"Target column {target_col!r} not found. Check spelling and capitalization.")

# Remove exact duplicate rows and rows with missing target labels.
before = len(df)
df = df.drop_duplicates().dropna(subset=[target_col]).copy()
print(f"Removed {before - len(df)} duplicate/missing-target rows. New shape: {df.shape}")

X = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].copy()

if X.shape[1] == 0:
    raise ValueError("No feature columns remain after separating the target.")

# Encode target labels for reliable classification metrics.
label_encoder = LabelEncoder()
y = pd.Series(label_encoder.fit_transform(y_raw.astype(str)), index=y_raw.index, name=target_col)
class_names = [str(x) for x in label_encoder.classes_]
print("Target classes:", class_names)
print("Class count:", len(class_names))
if y.nunique() < 2:
    raise ValueError("Classification requires at least two target classes.")


## 3. Exploratory data analysis (EDA)

In [ ]:

print("Data types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nNumerical summary:")
display(df.describe(include=[np.number]).T)

fig, ax = plt.subplots(figsize=(8, 4))
y.value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Target class distribution")
ax.set_xlabel("Encoded class")
ax.set_ylabel("Count")
ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
plt.tight_layout()
plt.show()

numeric_cols_eda = X.select_dtypes(include=np.number).columns.tolist()
if numeric_cols_eda:
    X[numeric_cols_eda[:12]].hist(figsize=(12, 8), bins=20)
    plt.suptitle("Numeric feature distributions (up to 12 columns)")
    plt.tight_layout()
    plt.show()

categorical_cols_eda = X.select_dtypes(exclude=np.number).columns.tolist()
for col in categorical_cols_eda[:5]:
    counts = X[col].astype(str).value_counts().head(15)
    ax = counts.plot(kind="bar", figsize=(8, 3), title=f"Top categories: {col}")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

if len(numeric_cols_eda) >= 2:
    plt.figure(figsize=(10, 7))
    corr = X[numeric_cols_eda].corr(numeric_only=True)
    plt.imshow(corr, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Numeric feature correlation matrix")
    plt.tight_layout()
    plt.show()


## 4. Train/test split and preprocessing
Gradient-boosted trees do not require feature scaling. Missing numeric values are median-imputed; missing categorical values are filled with the most frequent value, then one-hot encoded.

In [ ]:

class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError(
        "At least one class has fewer than 2 rows. Stratified train/test splitting and CV need more examples per class."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

transformers = []
if numeric_cols:
    transformers.append(("num", SimpleImputer(strategy="median"), numeric_cols))
if categorical_cols:
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", ohe)
    ])
    transformers.append(("cat", categorical_pipe, categorical_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

smallest_train_class = int(y_train.value_counts().min())
n_splits = min(5, smallest_train_class)
if n_splits < 2:
    raise ValueError("Not enough training examples in the smallest class for cross-validation.")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", len(numeric_cols), "| Categorical features:", len(categorical_cols))
print("Stratified CV folds:", n_splits)


## 5. Gradient Boosting + GridSearchCV
The grid below uses only the four cheat-sheet search dimensions. The optimization metric is macro F1, which gives each class equal weight.

In [ ]:

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__max_depth": [1, 3, 5],
    "model__subsample": [0.8, 1.0],
}
print("Expected parameter combinations:", 3 * 3 * 3 * 2)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
])

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring=scoring,
    refit="f1_macro",
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
    error_score="raise"
)

start_time = time.time()
grid_search.fit(X_train, y_train)
fit_seconds = time.time() - start_time
best_model = grid_search.best_estimator_

print(f"\nGrid search completed in {fit_seconds:.1f} seconds.")
print("Best parameters:")
display(pd.Series(grid_search.best_params_, name="best value").to_frame())
print(f"Best mean CV macro-F1: {grid_search.best_score_:.4f}")


## 6. Compare the grid-search candidates

In [ ]:

cv_results = pd.DataFrame(grid_search.cv_results_).sort_values("rank_test_f1_macro")
show_cols = [
    "rank_test_f1_macro", "mean_test_f1_macro", "std_test_f1_macro",
    "mean_test_accuracy", "mean_test_balanced_accuracy", "mean_fit_time",
    "param_model__n_estimators", "param_model__learning_rate",
    "param_model__max_depth", "param_model__subsample"
]
display(cv_results[show_cols].head(15).reset_index(drop=True))

top = cv_results.head(15).copy()
labels = [
    f"n={r['param_model__n_estimators']}, lr={r['param_model__learning_rate']}, "
    f"depth={r['param_model__max_depth']}, sub={r['param_model__subsample']}"
    for _, r in top.iterrows()
]
plt.figure(figsize=(11, 6))
plt.barh(range(len(top))[::-1], top["mean_test_f1_macro"].to_numpy())
plt.yticks(range(len(top))[::-1], labels)
plt.xlabel("Mean cross-validated macro F1")
plt.title("Top 15 Gradient Boosting configurations")
plt.tight_layout()
plt.show()


## 7. Final evaluation on the held-out test set

In [ ]:

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test) if hasattr(best_model, "predict_proba") else None

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "F1 (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "MCC": matthews_corrcoef(y_test, y_pred),
    "Cohen's kappa": cohen_kappa_score(y_test, y_pred),
}
if y_proba is not None:
    try:
        metrics["Log loss"] = log_loss(y_test, y_proba, labels=np.arange(len(class_names)))
    except Exception:
        pass
    try:
        if len(class_names) == 2:
            metrics["ROC-AUC"] = roc_auc_score(y_test, y_proba[:, 1])
            metrics["Average precision"] = average_precision_score(y_test, y_proba[:, 1])
        else:
            metrics["ROC-AUC (macro OVR)"] = roc_auc_score(
                y_test, y_proba, multi_class="ovr", average="macro",
                labels=np.arange(len(class_names))
            )
    except Exception as exc:
        print("Some probability-based metrics could not be calculated:", exc)

display(pd.Series(metrics, name="test score").to_frame())
print("\nClassification report:")
print(classification_report(
    y_test, y_pred, labels=np.arange(len(class_names)),
    target_names=class_names, zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Gradient Boosting — confusion matrix")
plt.tight_layout()
plt.show()


## 8. ROC and precision-recall curves (when applicable)

In [ ]:

if y_proba is not None and len(class_names) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    RocCurveDisplay.from_predictions(y_test, y_proba[:, 1], ax=axes[0])
    axes[0].set_title("ROC curve")
    PrecisionRecallDisplay.from_predictions(y_test, y_proba[:, 1], ax=axes[1])
    axes[1].set_title("Precision-recall curve")
    plt.tight_layout()
    plt.show()
elif y_proba is not None and len(class_names) > 2:
    print("Multiclass target detected. Macro one-vs-rest ROC-AUC is included in the metric table where calculable.")
else:
    print("Probability estimates are unavailable for this model/configuration.")


## 9. Feature importance

In [ ]:

fitted_preprocessor = best_model.named_steps["preprocess"]
fitted_classifier = best_model.named_steps["model"]

try:
    feature_names = fitted_preprocessor.get_feature_names_out()
    importances = fitted_classifier.feature_importances_
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)
    display(importance_df.head(20))
    top_imp = importance_df.head(20).sort_values("importance")
    plt.figure(figsize=(9, 6))
    plt.barh(top_imp["feature"], top_imp["importance"])
    plt.xlabel("Feature importance")
    plt.title("Top 20 transformed features")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Could not map feature importance to transformed feature names:", exc)


## 10. Learning curve
This shows whether performance may improve with more training data. It can take additional time because each point fits multiple models.

In [ ]:

train_sizes, train_scores, validation_scores = learning_curve(
    estimator=best_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring="f1_macro",
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=-1,
    shuffle=True,
    random_state=RANDOM_STATE
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = validation_scores.mean(axis=1)
val_std = validation_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker="o", label="Training macro-F1")
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15)
plt.plot(train_sizes, val_mean, marker="o", label="CV macro-F1")
plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.15)
plt.xlabel("Number of training examples")
plt.ylabel("Macro F1")
plt.title("Learning curve")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Optional paired significance check: tuned model vs default model
This compares fold-level macro-F1 for the tuned model against a default Gradient Boosting model on the training set. With only a few CV folds, statistical tests have low power; treat them as a rough diagnostic, not definitive proof. The selected best model was chosen using these data, so the comparison is also subject to model-selection optimism.

In [ ]:

default_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
])

tuned_scores = cross_validate(
    best_model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1
)["test_score"]
default_scores = cross_validate(
    default_model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1
)["test_score"]

print("Tuned model fold macro-F1:", np.round(tuned_scores, 4))
print("Default model fold macro-F1:", np.round(default_scores, 4))
print("Mean tuned:", round(float(np.mean(tuned_scores)), 4))
print("Mean default:", round(float(np.mean(default_scores)), 4))
try:
    t_stat, t_p = ttest_rel(tuned_scores, default_scores)
    print(f"Paired t-test: statistic={t_stat:.4f}, p-value={t_p:.4f}")
except Exception as exc:
    print("Paired t-test unavailable:", exc)
try:
    if np.allclose(tuned_scores, default_scores):
        print("Wilcoxon test skipped because paired scores are identical.")
    else:
        w_stat, w_p = wilcoxon(tuned_scores, default_scores)
        print(f"Wilcoxon signed-rank test: statistic={w_stat:.4f}, p-value={w_p:.4f}")
except Exception as exc:
    print("Wilcoxon test unavailable:", exc)


## 12. Export results

In [ ]:

# Save model-selection results, held-out predictions, and summary metrics.
cv_results.to_csv("gradient_boosting_grid_search_results.csv", index=False)

prediction_df = pd.DataFrame({
    "actual_encoded": y_test.to_numpy(),
    "predicted_encoded": y_pred,
    "actual_label": label_encoder.inverse_transform(y_test.to_numpy()),
    "predicted_label": label_encoder.inverse_transform(np.asarray(y_pred, dtype=int))
}, index=X_test.index)
prediction_df.to_csv("gradient_boosting_test_predictions.csv", index=True)

summary = pd.DataFrame([{
    "model": "GradientBoostingClassifier",
    "best_cv_macro_f1": grid_search.best_score_,
    "test_accuracy": metrics.get("Accuracy", np.nan),
    "test_balanced_accuracy": metrics.get("Balanced accuracy", np.nan),
    "test_macro_f1": metrics.get("F1 (macro)", np.nan),
    "fit_seconds": fit_seconds,
    "best_params": str(grid_search.best_params_)
}])
summary.to_csv("gradient_boosting_summary.csv", index=False)

print("Saved:")
print("- gradient_boosting_grid_search_results.csv")
print("- gradient_boosting_test_predictions.csv")
print("- gradient_boosting_summary.csv")
display(summary)


## Quick viva notes
- **Gradient Boosting:** builds an additive ensemble sequentially; each new tree attempts to improve errors made by the current ensemble.
- **`n_estimators`:** number of boosting stages/trees.
- **`learning_rate`:** shrinks each tree's contribution; smaller values often need more trees.
- **`max_depth`:** controls the depth/complexity of each individual tree.
- **`subsample`:** fraction of training rows used for each boosting stage. Values below `1.0` introduce stochastic gradient boosting.
- **Why split before GridSearchCV?** Hyperparameter selection uses only training data; the held-out test set is reserved for final evaluation.
- **Why macro-F1?** It averages F1 across classes equally, which is useful when class sizes differ.
- **Why no scaling?** Tree splits are generally invariant to monotonic feature scaling.
- **What to change in the practical?** CSV path, exact target column name, and—if instructed by your faculty—the scoring metric or CV fold count. Keep the cheat-sheet grid values unchanged unless the question specifies otherwise.